In [ ]:
from google.colab import files
uploaded = files.upload()  # select your v2_densenet_transfer.keras file
print("Uploaded:", list(uploaded.keys()))

In [ ]:
import tensorflow as tf

model_filename = list(uploaded.keys())[0]
model = tf.keras.models.load_model(model_filename)

# Retrieve by TYPE rather than a hardcoded name — Keras can auto-suffix
# names (e.g. "densenet121_1") depending on session history, so matching
# by class is safer than assuming the exact string.
base_model = next(l for l in model.layers if l.__class__.__name__ == "Functional")
gap_layer = next(l for l in model.layers if l.__class__.__name__ == "GlobalAveragePooling2D")
dense_layer = next(l for l in model.layers if l.__class__.__name__ == "Dense")

print("Retrieved:", base_model.name, gap_layer.name, dense_layer.name)
model.summary()

In [ ]:
def make_gradcam_heatmap(img_array, base_model, gap_layer, dense_layer, class_index, last_conv_layer_name="relu"):
    """
    Built from base_model's OWN standalone input/output graph
    """
    grad_model = tf.keras.models.Model(
        inputs=base_model.input,
        outputs=[base_model.get_layer(last_conv_layer_name).output, base_model.output],
    )

    with tf.GradientTape() as tape:
        conv_output, base_features = grad_model(img_array)
        pooled = gap_layer(base_features)
        predictions = dense_layer(pooled)
        class_output = predictions[:, class_index]

    grads = tape.gradient(class_output, conv_output)
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))

    conv_output = conv_output[0]
    heatmap = conv_output @ pooled_grads[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap)
    heatmap = tf.maximum(heatmap, 0) / (tf.math.reduce_max(heatmap) + 1e-8)
    return heatmap.numpy()

In [ ]:
import numpy as np
from PIL import Image
import io, base64

def overlay_heatmap(original_image_array, heatmap, alpha=0.4):
    h, w = original_image_array.shape[:2]
    heatmap_img = Image.fromarray(np.uint8(heatmap * 255)).resize((w, h), resample=Image.BILINEAR)
    heatmap_resized = np.array(heatmap_img) / 255.0

    heatmap_colored = np.zeros((h, w, 3), dtype=np.float32)
    heatmap_colored[..., 0] = heatmap_resized  # simple red-intensity overlay, no matplotlib dependency

    original_uint8 = (original_image_array * 255).astype(np.float32)
    overlay = original_uint8 * (1 - alpha) + heatmap_colored * 255 * alpha
    overlay = np.clip(overlay, 0, 255).astype(np.uint8)

    buf = io.BytesIO()
    Image.fromarray(overlay).save(buf, format="PNG")
    return base64.b64encode(buf.getvalue()).decode("utf-8")

In [ ]:
import kagglehub
kagglehub.login()

sample_path = kagglehub.dataset_download("nih-chest-xrays/sample")

import os
img_dir_candidates = [root for root, dirs, files in os.walk(sample_path) if any(f.lower().endswith(".png") for f in files)]
SAMPLE_IMAGES_DIR = img_dir_candidates[0]

sample_files = os.listdir(SAMPLE_IMAGES_DIR)[:5]
print("Testing on:", sample_files)

In [ ]:
import matplotlib.pyplot as plt

CONDITIONS = [
    "Atelectasis", "Cardiomegaly", "Effusion", "Infiltration", "Mass",
    "Nodule", "Pneumonia", "Pneumothorax", "Consolidation", "Edema",
    "Emphysema", "Fibrosis", "Pleural_Thickening", "Hernia"
]

fig, axes = plt.subplots(1, len(sample_files), figsize=(4 * len(sample_files), 4))

for ax, filename in zip(axes, sample_files):
    img = Image.open(os.path.join(SAMPLE_IMAGES_DIR, filename)).convert("RGB").resize((224, 224))
    img_array = np.asarray(img, dtype=np.float32) / 255.0
    batch = np.expand_dims(img_array, axis=0)

    predictions = model.predict(batch, verbose=0)[0]
    top_class_index = int(np.argmax(predictions))
    top_condition = CONDITIONS[top_class_index]
    top_prob = predictions[top_class_index]

    heatmap = make_gradcam_heatmap(batch, base_model, gap_layer, dense_layer, top_class_index)
    overlay_b64 = overlay_heatmap(img_array, heatmap)
    overlay_img = Image.open(io.BytesIO(base64.b64decode(overlay_b64)))

    ax.imshow(overlay_img)
    ax.set_title(f"{top_condition}\n{top_prob:.2f}")
    ax.axis("off")

plt.tight_layout()
plt.show()